# Langchain Semantic Search with Precise Page Grounding

This notebook demonstrates how to merge section data while preserving page numbers, use HuggingFace `all-MiniLM-L6-v2` embeddings, split the text using `SemanticChunker`, and integrate with ChromaDB via Langchain.

In [14]:
import json
import re
from collections import defaultdict

from langchain.schema import Document
from langchain.vectorstores import Chroma
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings


### 1. Load Data
We load the modular section records (read-only) to prepare for merging.

In [15]:
data_path = "../Data/kaggle_section_records_modular.json"
with open(data_path, "r", encoding="utf-8") as f:
    records = json.load(f)

print(f"Loaded {len(records)} records.")

Loaded 764 records.


### 2. Merge Text with Page Markers
We group the text by `heading` and `sub_heading`. To preserve accurate page grounding, we inject `[PAGE X]` markers directly into the concatenated text. When the `SemanticChunker` breaks this text apart, the markers will naturally remain attached to the sentences they correspond to.

In [16]:
merged_sections = defaultdict(list)

# Group records by their section identity
for r in records:
    meta = r["metadata"]
    section_id = f"{meta['heading']} ||| {meta['sub_heading']}"
    merged_sections[section_id].append(r)

annotated_documents = []

for section_id, parts in merged_sections.items():
    parts.sort(key=lambda x: int(x["metadata"]["page"]))
    
    heading, sub_heading = section_id.split(" ||| ")
    combined_text = ""
    
    for part in parts:
        page_num = part["metadata"]["page"]
        # Inject a clear marker into the text so the chunker keeps it
        combined_text += f" [PAGE {page_num}] " + part["context"]
        
    # Clean up whitespace
    combined_text = re.sub(r"\s+", " ", combined_text).strip()
    
    annotated_documents.append({
        "heading": heading,
        "sub_heading": sub_heading,
        "annotated_text": combined_text
    })
    
print(f"Merged into {len(annotated_documents)} unique sections.")

Merged into 185 unique sections.


### 3. Initialize HuggingFace Embeddings & Semantic Chunker
We use `all-MiniLM-L6-v2`. The `SemanticChunker` uses this model to find points in the text where the topic shifts logically.

In [17]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
chunker = SemanticChunker(embeddings, breakpoint_threshold_type="percentile")

### 4. Extract Chunks and Restore Page Metadata
After chunking, we will use regex to find all the `[PAGE X]` markers within a given chunk. We will extract the lowest and highest page numbers found as `start_page` and `end_page`. We then strip the markers from the final text so they don't pollute the prompt later.

In [26]:
langchain_docs = []
page_marker_re = re.compile(r"\[PAGE (\d+)\]")

import warnings
warnings.filterwarnings("ignore") # Ignore token length warnings from SemanticChunker

# Process a subset initially to demonstrate if needed, or all of them
for doc in annotated_documents:
    text = doc["annotated_text"]
    
    # Split the long section into semantic blocks
    chunks = chunker.split_text(text)
    
    current_page = "Unknown"
    for chunk in chunks:
        # Extract all page numbers mentioned in this specific chunk
        pages_found = [int(p) for p in page_marker_re.findall(chunk)]
        
        if pages_found:
            start_page = str(min(pages_found))
            end_page = str(max(pages_found))
            current_page = end_page
        else:
            start_page = current_page
            end_page = current_page
            
        # Clean the markers out of the final chunk text
        clean_chunk = page_marker_re.sub("", chunk).strip()
        clean_chunk = re.sub(r"\s+", " ", clean_chunk).strip()
        
        # Skip empty chunks
        if not clean_chunk:
            continue
            
        metadata = {
            "heading": doc["heading"],
            "sub_heading": doc["sub_heading"],
            "start_page": start_page,
            "end_page": end_page
        }
        langchain_docs.append(Document(page_content=clean_chunk, metadata=metadata))

print(f"Generated {len(langchain_docs)} semantic chunks with precise page grounding.")


Generated 983 semantic chunks with precise page grounding.


### 5. Build the Chroma VectorStore
Load the standard Langchain `Document` objects into a persistent Chroma directory.

In [ ]:
persist_directory = "./chroma_langchain_db"

# Clean up existing database so we don't accidentally duplicate chunks if we run this cell twice
import shutil
import os
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

# Create the vector store fresh
vectorstore = Chroma.from_documents(
    documents=langchain_docs,
    embedding=embeddings,
    persist_directory=persist_directory
)
vectorstore.persist()
print("Vector store successfully built and persisted!")

Vector store successfully built and persisted!


### 6. Test Semantic Retrieval
Let's test retrieving documents for a specific concept to see our metadata at work.

In [30]:
from langchain.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Initialize the identical embedding model used during creation
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Load the vector store directly from disk (no need to run previous cells!)
persist_directory = "./chroma_langchain_db"
vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
print(vectorstore._collection.count())


983


In [31]:
# 3. Run semantic retrievals independently
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
query = "What are the psychological effects of sleep deprivation?"
results = retriever.get_relevant_documents(query)

for i, res in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(f"Heading: {res.metadata['heading']}")
    print(f"Sub-heading: {res.metadata['sub_heading']}")
    print(f"Pages: {res.metadata['start_page']} to {res.metadata['end_page']}")
    print(f"Context Snippet: {res.page_content[:200]}...")


--- Result 1 ---
Heading: States of Consciousness
Sub-heading: Personal Application Questions
Pages: 143 to 144
Context Snippet: 4.2 Sleep and Why We Sleep 4.3 Stages of Sleep 4.4 Sleep Problems and Disorders 4.5 Substance Use and Abuse 4.6 Other States of Consciousness Personal Application Questions 38. We experience shifts in...

--- Result 2 ---
Heading: States of Consciousness
Sub-heading: Summary
Pages: 139 to 139
Context Snippet: Summary 4.1 What Is Consciousness? States of consciousness vary over the course of the day and throughout our lives. Important factors in these changes are the biological rhythms, and, more specifical...

--- Result 3 ---
Heading: States of Consciousness
Sub-heading: Critical Thinking Questions
Pages: 143 to 143
Context Snippet: 26. Generally, humans are considered diurnal which means we are awake during the day and asleep during the night. Many rodents, on the other hand, are nocturnal. Why do you think different animals hav...
